# NorthStar Python and MongoDB Solution

This notebook performs Python-based cleaning and analytical feature engineering, then shows how the semi-structured records can be remodelled into MongoDB Atlas.


In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('/content/northstar_dataset') if Path('/content/northstar_dataset').exists() else Path('/Users/danishkayani/Documents/Codex/2026-05-23/files-mentioned-by-the-user-msg/northstar_dataset')

orders = pd.read_csv(DATA_DIR / 'orders.csv')
deliveries = pd.read_csv(DATA_DIR / 'deliveries.csv')
complaints = pd.read_csv(DATA_DIR / 'complaints.csv')
vehicles = pd.read_csv(DATA_DIR / 'vehicles.csv')
hubs = pd.read_csv(DATA_DIR / 'hubs.csv')
app_events = pd.read_csv(DATA_DIR / 'app_events.csv')
incidents = pd.read_csv(DATA_DIR / 'incidents.csv')


In [2]:
def clean_zone(series):
    mapping = {
        'NORTH': 'North',
        'SOUTH': 'South',
        'EAST': 'East',
        'WEST': 'West',
        'AIRPORT': 'Airport',
        'CENTRAL': 'Central',
        'CTR': 'Central',
        'RIVERSIDE': 'Riverside'
    }
    return series.fillna('').astype(str).str.strip().str.upper().map(mapping).fillna(series)

orders['pickup_zone_clean'] = clean_zone(orders['pickup_zone'])
orders['dropoff_zone_clean'] = clean_zone(orders['dropoff_zone'])
hubs['zone_clean'] = clean_zone(hubs['zone'])
app_events['zone_context_clean'] = clean_zone(app_events['zone_context'])

deliveries['dispatch_time'] = pd.to_datetime(deliveries['dispatch_time'])
deliveries['delivery_completed_at'] = pd.to_datetime(deliveries['delivery_completed_at'], errors='coerce')
deliveries['completion_hours'] = (deliveries['delivery_completed_at'] - deliveries['dispatch_time']).dt.total_seconds() / 3600


In [3]:
merged = deliveries.merge(orders, on='order_id', how='left') \
    .merge(hubs[['hub_id', 'hub_name', 'zone_clean']], on='hub_id', how='left') \
    .merge(vehicles[['vehicle_id', 'maintenance_status', 'battery_health_pct']], on='vehicle_id', how='left')

complaint_totals = complaints.groupby('order_id', as_index=False).agg(
    complaint_count=('complaint_id', 'count'),
    compensation_amount=('compensation_amount', 'sum')
)

merged = merged.merge(complaint_totals, on='order_id', how='left')
merged['complaint_count'] = merged['complaint_count'].fillna(0)
merged['compensation_amount'] = merged['compensation_amount'].fillna(0)
merged['net_after_comp'] = merged['order_value'] - merged['fuel_or_charge_cost'] - merged['compensation_amount']

merged[['delivery_status', 'hub_name', 'maintenance_status', 'net_after_comp']].head()


,delivery_status,hub_name,maintenance_status,net_after_comp
0,Failed,Central Core,Active,139.09
1,OnTime,South Link,Active,-3.37
2,OnTime,South Link,Active,133.42
3,Delayed,South Link,Active,-2.51
4,OnTime,North Exchange,InRepair,66.80


In [4]:
proof_risk = merged.groupby('proof_of_completion_missing', as_index=False).agg(
    deliveries=('delivery_id', 'count'),
    delayed_pct=('delivery_status', lambda s: round((s.eq('Delayed').mean()) * 100, 1)),
    failed_pct=('delivery_status', lambda s: round((s.eq('Failed').mean()) * 100, 1)),
    avg_rating=('customer_rating_post_delivery', 'mean')
)

maintenance_risk = merged.groupby('maintenance_status', as_index=False).agg(
    deliveries=('delivery_id', 'count'),
    ontime_pct=('delivery_status', lambda s: round((s.eq('OnTime').mean()) * 100, 1)),
    failed_pct=('delivery_status', lambda s: round((s.eq('Failed').mean()) * 100, 1)),
    avg_battery_health=('battery_health_pct', 'mean')
)

proof_risk, maintenance_risk


(   proof_of_completion_missing  deliveries  delayed_pct  failed_pct  \
 0                            0         881         17.8        12.3   
 1                            1          69         65.2        34.8   
 
    avg_rating  
 0    3.919263  
 1    3.167941  ,
   maintenance_status  deliveries  ontime_pct  failed_pct  avg_battery_health
 0             Active         542        70.8         8.3           76.561896
 1           InRepair         254        49.2        30.3           76.725984
 2          Scheduled         154        69.5         6.5           78.735714)

In [5]:
hub_profitability = merged.groupby('hub_name', as_index=False).agg(
    deliveries=('delivery_id', 'count'),
    avg_order_value=('order_value', 'mean'),
    avg_direct_cost=('fuel_or_charge_cost', 'mean'),
    avg_compensation=('compensation_amount', 'mean'),
    avg_net_after_comp=('net_after_comp', 'mean')
).sort_values('avg_net_after_comp')

hub_profitability


,hub_name,deliveries,avg_order_value,avg_direct_cost,avg_compensation,avg_net_after_comp
0,Airport Hub,104,86.326827,13.319231,4.305481,68.702115
5,Riverside Hub,115,89.932435,12.922087,5.821826,71.188522
3,Midtown Relay,128,88.723906,11.708203,5.806797,71.208906
7,West Gate,127,90.064409,13.167008,4.411496,72.485906
4,North Exchange,136,91.753309,12.755809,3.961691,75.035809
6,South Link,106,90.354811,12.565000,2.195943,75.593868
1,Central Core,115,99.757739,13.686000,6.023130,80.048609
2,East Dock,119,97.684958,12.744202,4.683277,80.257479


## MongoDB Atlas Design

The app events, complaint histories, incident chains, and evolving case notes are good candidates for a document model. The collection below keeps customer, order, complaint, and event history together.


In [6]:
customer_case_document = {
    '_id': 'CASE_O00814',
    'order_id': 'O00814',
    'customer': {
        'customer_id': 'C0464',
        'home_zone': 'North',
        'customer_type': 'Consumer',
        'loyalty_score': 58.4
    },
    'service_order': {
        'service_type': 'Passenger',
        'pickup_zone': 'Central',
        'dropoff_zone': 'Airport',
        'priority_level': 'High',
        'order_value': 94.50,
        'booking_channel': 'App'
    },
    'delivery_summary': {
        'delivery_id': 'DL00481',
        'hub_id': 'H04',
        'hub_name': 'Central Core',
        'delivery_status': 'Delayed',
        'manual_route_override_count': 2,
        'proof_of_completion_missing': False
    },
    'complaints': [
        {
            'complaint_id': 'CP0001',
            'complaint_type': 'AppIssue',
            'severity': 'High',
            'status': 'Open',
            'compensation_amount': 23.99
        }
    ],
    'app_events': [
        {
            'event_id': 'AE00999',
            'event_type': 'track_order',
            'api_latency_ms': 611,
            'success_flag': True
        }
    ],
    'case_status': 'Open'
}

customer_case_document


{'_id': 'CASE_O00814',
 'order_id': 'O00814',
 'customer': {'customer_id': 'C0464',
  'home_zone': 'North',
  'customer_type': 'Consumer',
  'loyalty_score': 58.4},
 'service_order': {'service_type': 'Passenger',
  'pickup_zone': 'Central',
  'dropoff_zone': 'Airport',
  'priority_level': 'High',
  'order_value': 94.5,
  'booking_channel': 'App'},
 'delivery_summary': {'delivery_id': 'DL00481',
  'hub_id': 'H04',
  'hub_name': 'Central Core',
  'delivery_status': 'Delayed',
  'manual_route_override_count': 2,
  'proof_of_completion_missing': False},
 'complaints': [{'complaint_id': 'CP0001',
   'complaint_type': 'AppIssue',
   'severity': 'High',
   'status': 'Open',
   'compensation_amount': 23.99}],
 'app_events': [{'event_id': 'AE00999',
   'event_type': 'track_order',
   'api_latency_ms': 611,
   'success_flag': True}],
 'case_status': 'Open'}

In [7]:
from pymongo import MongoClient, ASCENDING, DESCENDING

MONGODB_URI = os.getenv('MONGODB_URI')
if MONGODB_URI:
    client = MongoClient(MONGODB_URI)
else:
    import mongomock
    client = mongomock.MongoClient()
db = client['northstar']
customer_cases = db['customer_cases']
delivery_events = db['delivery_events']

customer_cases.delete_many({})
delivery_events.delete_many({})
customer_cases.insert_one(customer_case_document)
delivery_events.insert_one({
    'delivery_id': 'DL00481',
    'vehicle_id': 'V056',
    'maintenance_status': 'Active',
    'event_timeline': [
        {'event_type': 'dispatch', 'event_timestamp': '2025-03-29T22:10:00Z'},
        {'event_type': 'route_override', 'event_timestamp': '2025-03-29T22:22:00Z'}
    ]
})

customer_cases.create_index([('order_id', ASCENDING)], unique=True)
customer_cases.create_index([('customer.customer_id', ASCENDING), ('case_status', ASCENDING)])
customer_cases.create_index([('delivery_summary.hub_id', ASCENDING), ('delivery_summary.delivery_status', ASCENDING)])
customer_cases.create_index([('complaints.status', ASCENDING), ('complaints.severity', ASCENDING)])
customer_cases.create_index([('app_events.event_timestamp', DESCENDING)])

delivery_events.create_index([('delivery_id', ASCENDING)], unique=True)
delivery_events.create_index([('vehicle_id', ASCENDING), ('maintenance_status', ASCENDING)])


'vehicle_id_1_maintenance_status_1'

In [8]:
pipeline = [
    {
        '$match': {
            'delivery_summary.hub_id': 'H04',
            'delivery_summary.delivery_status': {'$in': ['Delayed', 'Failed']},
            'case_status': 'Open'
        }
    },
    {
        '$project': {
            '_id': 0,
            'order_id': 1,
            'customer.customer_id': 1,
            'delivery_summary.delivery_status': 1,
            'complaints': 1
        }
    }
]

list(customer_cases.aggregate(pipeline))

query_cursor = customer_cases.find(
    {
        'delivery_summary.hub_id': 'H04',
        'delivery_summary.delivery_status': {'$in': ['Delayed', 'Failed']},
        'case_status': 'Open'
    }
)

if hasattr(query_cursor, 'explain'):
    query_cursor.explain('executionStats')
else:
    {'note': 'explain() is available against a live MongoDB server or Atlas cluster, but not in the local fallback client.'}


## Interpretation

Python is used here to connect structured and semi-structured evidence into one analytical view. MongoDB Atlas then becomes the operational store for customer case histories, event sequences, and exception tracking, with indexes designed around the case study's likely access patterns.
